# SCSO seed sensitivity

Evaluate five independent SCSO seeds (42, 52, 62, 72, and 82), keeping the primary training/test split and cross-validation folds fixed at seed 42.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_FILE = PROJECT_ROOT / "data" / "gallstone.xlsx"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Download the UCI Gallstone dataset and save it to {DATA_FILE}")


In [ ]:
from __future__ import annotations

import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


RANDOM_SEED = 42

DATA_PATH = str(DATA_FILE)
TARGET_COL = "Gallstone Status"


POSITIVE_LABEL = 0  # verified: raw 0 = gallstone present; raw 1 = absent


SAMPLE_ID_COL = None

N_SPLITS = 5
TEST_SIZE = 0.20

HPO_BUDGET = 100
SCSO_POP_SIZE = 10
SCSO_S_MAX = 2.0

OUTPUT_DIR = RESULTS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


SIGN_EPS = 1e-12


FEATURE_GROUPS = {
    "demographic": [
        "Age",
        "Gender",
        "Height",
        "Weight",
        "Body Mass Index (BMI)",
    ],
    "clinical": [
        "Comorbidity",
        "Coronary Artery Disease (CAD)",
        "Hypothyroidism",
        "Hyperlipidemia",
        "Diabetes Mellitus (DM)",
        "Hepatic Fat Accumulation (HFA)",
    ],
    "bioimpedance": [
        "Total Body Water (TBW)",
        "Extracellular Water (ECW)",
        "Intracellular Water (ICW)",
        "Extracellular Fluid/Total Body Water (ECF/TBW)",
        "Total Body Fat Ratio (TBFR) (%)",
        "Lean Mass (LM) (%)",
        "Body Protein Content (Protein) (%)",
        "Visceral Fat Rating (VFR)",
        "Bone Mass (BM)",
        "Muscle Mass (MM)",
        "Obesity (%)",
        "Total Fat Content (TFC)",
        "Visceral Fat Area (VFA)",
        "Visceral Muscle Area (VMA) (Kg)",
    ],
    "laboratory": [
        "Glucose",
        "Total Cholesterol (TC)",
        "Low Density Lipoprotein (LDL)",
        "High Density Lipoprotein (HDL)",
        "Triglyceride",
        "Aspartat Aminotransferaz (AST)",
        "Alanin Aminotransferaz (ALT)",
        "Alkaline Phosphatase (ALP)",
        "Creatinine",
        "Glomerular Filtration Rate (GFR)",
        "C-Reactive Protein (CRP)",
        "Hemoglobin (HGB)",
        "Vitamin D",
    ],
}


SCENARIOS = {
    "A_baseline": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
    ),
    "B_plus_bia": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["bioimpedance"]
    ),
    "C_plus_lab": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["laboratory"]
    ),
    "D_full": (
        FEATURE_GROUPS["demographic"]
        + FEATURE_GROUPS["clinical"]
        + FEATURE_GROUPS["bioimpedance"]
        + FEATURE_GROUPS["laboratory"]
    ),
}


SCENARIO_ORDER = ["A_baseline", "B_plus_bia", "C_plus_lab", "D_full"]
MODEL_ORDER = ["LR", "SVM", "RF", "XGB"]
HPO_ORDER = ["GridSearch", "SCSO"]
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]


def make_pipeline(estimator) -> Pipeline:
    """Same preprocessing structure for every classifier."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", estimator),
        ]
    )


MODEL_SPECS = {
    "LR": {

        "estimator": LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_SEED,
        ),

        "grid": {
            "clf__C": np.logspace(-3, 2, 100),
        },

        "scso_spec": {
            "clf__C": (-3.0, 2.0, "log_cont"),
        },
    },
    "SVM": {
        "estimator": SVC(
            kernel="rbf",
            probability=False,
            random_state=RANDOM_SEED,
        ),

        "grid": {
            "clf__C": np.logspace(-2, 2, 10),
            "clf__gamma": np.logspace(-4, 0, 10),
        },

        "scso_spec": {
            "clf__C": (-2.0, 2.0, "log_cont"),
            "clf__gamma": (-4.0, 0.0, "log_cont"),
        },
    },
    "RF": {
        "estimator": RandomForestClassifier(
            criterion="gini",
            bootstrap=True,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),

        "grid": {
            "clf__n_estimators": [50, 100, 150, 225, 300],
            "clf__max_depth": [5, 10, 15, 20, 25],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2],
        },
        "scso_spec": {
            "clf__n_estimators": (50.0, 300.0, "int"),
            "clf__max_depth": (5.0, 25.0, "int"),
            "clf__min_samples_split": (2.0, 5.0, "int"),
            "clf__min_samples_leaf": (1.0, 2.0, "int"),
        },
    },
    "XGB": {
        "estimator": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0,
        ),

        "grid": {
            "clf__n_estimators": [50, 100, 150, 225, 300],
            "clf__max_depth": [2, 3, 4, 6, 8],
            "clf__learning_rate": [0.01, 0.05, 0.10, 0.20],
        },
        "scso_spec": {
            "clf__n_estimators": (50.0, 300.0, "int"),
            "clf__max_depth": (2.0, 8.0, "int"),
            "clf__learning_rate": (0.01, 0.20, "cont"),
        },
    },
}


def to_builtin(value: Any) -> Any:
    """Convert numpy/pandas scalar values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if np.isscalar(value) else False:
        return None
    return value


def params_to_json(params: dict[str, Any]) -> str:
    return json.dumps(to_builtin(params), sort_keys=True)


def load_dataset(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Dataset not found: {p.resolve()}\n"
            f"Set DATA_PATH at the top of this script."
        )

    suffix = p.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(p)
    if suffix == ".csv":
        return pd.read_csv(p)

    raise ValueError(
        f"Unsupported file type '{suffix}'. Use .xlsx, .xls, or .csv."
    )


def validate_columns_and_data(df: pd.DataFrame) -> None:
    all_features = [c for cols in FEATURE_GROUPS.values() for c in cols]

    duplicated_feature_names = sorted(
        {c for c in all_features if all_features.count(c) > 1}
    )
    if duplicated_feature_names:
        raise ValueError(
            f"Duplicate feature names found across FEATURE_GROUPS: "
            f"{duplicated_feature_names}"
        )

    missing_features = [c for c in all_features if c not in df.columns]
    if missing_features:
        raise ValueError(
            "The following feature columns are missing from the dataset:\n"
            + "\n".join(f" - {c}" for c in missing_features)
            + "\n\nColumns actually found:\n"
            + "\n".join(f" - {c}" for c in df.columns)
        )

    if TARGET_COL not in df.columns:
        raise ValueError(
            f"TARGET_COL='{TARGET_COL}' was not found.\n"
            f"Available columns: {list(df.columns)}"
        )

    if SAMPLE_ID_COL is not None and SAMPLE_ID_COL not in df.columns:
        raise ValueError(
            f"SAMPLE_ID_COL='{SAMPLE_ID_COL}' was not found."
        )

    if df[TARGET_COL].isna().any():
        raise ValueError("Target column contains missing values.")

    feature_missing = df[all_features].isna().sum()
    feature_missing = feature_missing[feature_missing > 0]
    if len(feature_missing):
        raise ValueError(
            "Feature columns contain missing values, but the manuscript "
            "specifies no imputation. Missing counts:\n"
            + feature_missing.to_string()
        )


    non_numeric = [
        c for c in all_features
        if not pd.api.types.is_numeric_dtype(df[c])
    ]
    if non_numeric:
        raise TypeError(
            "These predictor columns are not numeric, while the current "
            "manuscript specifies no additional encoding:\n"
            + "\n".join(f" - {c}" for c in non_numeric)
        )

    expected_counts = {
        "demographic": 5,
        "clinical": 6,
        "bioimpedance": 14,
        "laboratory": 13,
    }
    for group, expected in expected_counts.items():
        actual = len(FEATURE_GROUPS[group])
        if actual != expected:
            raise ValueError(
                f"Feature group '{group}' has {actual} variables; "
                f"expected {expected}."
            )

    if len(set(all_features)) != 38:
        raise ValueError(
            f"Expected 38 unique predictors, found {len(set(all_features))}."
        )


def encode_binary_target(raw_target: pd.Series) -> tuple[pd.Series, dict[str, Any]]:
    values = list(pd.unique(raw_target))

    print("\nRaw target distribution:")
    print(raw_target.value_counts(dropna=False))

    if len(values) != 2:
        raise ValueError(
            f"Expected a binary target, found {len(values)} unique values: {values}"
        )

    if POSITIVE_LABEL is None:
        raise ValueError(
            "\nPOSITIVE_LABEL is still None.\n"
            "Set POSITIVE_LABEL at the top of the script to the RAW target value "
            "that means 'has gallstone', then rerun the experiment.\n"
            f"Observed raw labels: {values}"
        )

    if POSITIVE_LABEL not in values:
        raise ValueError(
            f"POSITIVE_LABEL={POSITIVE_LABEL!r} is not present in target values "
            f"{values}."
        )

    negative_values = [v for v in values if v != POSITIVE_LABEL]
    negative_label = negative_values[0]

    y = (raw_target == POSITIVE_LABEL).astype(int)

    mapping_info = {
        "raw_positive_label": to_builtin(POSITIVE_LABEL),
        "raw_negative_label": to_builtin(negative_label),
        "encoded_positive": 1,
        "encoded_negative": 0,
    }

    print("\nConfirmed target mapping:")
    print(f"  Positive class (1) = raw label {POSITIVE_LABEL!r}")
    print(f"  Negative class (0) = raw label {negative_label!r}")
    print("\nEncoded target distribution:")
    print(y.value_counts().sort_index())

    return y, mapping_info


def get_sample_ids(df: pd.DataFrame) -> pd.Series:
    if SAMPLE_ID_COL is not None:
        return df[SAMPLE_ID_COL].astype(str)
    return pd.Series(df.index.astype(str), index=df.index, name="sample_id")


def continuous_score(fitted_pipeline: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, str]:
    """
    Continuous score for ROC-AUC:
    - probability for LR/RF/XGB
    - decision_function for SVM
    """
    clf = fitted_pipeline.named_steps["clf"]

    if hasattr(clf, "predict_proba"):
        score = fitted_pipeline.predict_proba(X)[:, 1]
        return np.asarray(score, dtype=float), "probability"

    if hasattr(clf, "decision_function"):
        score = fitted_pipeline.decision_function(X)
        return np.asarray(score, dtype=float), "decision_function"

    raise AttributeError(
        f"Classifier {clf.__class__.__name__} provides neither "
        f"predict_proba nor decision_function."
    )


def compute_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(
            precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y_true, y_score)),
    }


def summarize_fold_metrics(fold_df: pd.DataFrame) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    for metric in METRICS:
        vals = fold_df[metric].to_numpy(dtype=float)
        out[metric] = {
            "mean": float(np.mean(vals)),
            "sd": float(np.std(vals, ddof=1)),
        }
    return out


@dataclass
class CandidateEvaluation:
    eval_id: int
    params: dict[str, Any]
    fitness: float


class SCSO:
    """
    Sand Cat Swarm Optimization for maximizing CV Accuracy.

    Population update is synchronous and uses:
    - full population replacement after each update round,
    - no extra greedy acceptance rule,
    - random population member in exploration,
    - global best in exploitation,
    - roulette-wheel selection over 360 possible movement directions.

    Search coordinates are the encoded parameter domains themselves.
    """

    def __init__(
        self,
        param_spec: dict[str, tuple[float, float, str]],
        fitness_fn,
        pop_size: int = 10,
        max_evals: int = 100,
        s_max: float = 2.0,
        seed: int = RANDOM_SEED,
    ):
        if max_evals < pop_size:
            raise ValueError("max_evals must be >= pop_size.")
        if (max_evals - pop_size) % pop_size != 0:
            raise ValueError(
                "For an exact full-population budget, "
                "(max_evals - pop_size) must be divisible by pop_size."
            )

        self.param_spec = param_spec
        self.names = list(param_spec.keys())
        self.dim = len(self.names)

        self.lower = np.array(
            [param_spec[n][0] for n in self.names], dtype=float
        )
        self.upper = np.array(
            [param_spec[n][1] for n in self.names], dtype=float
        )

        self.fitness_fn = fitness_fn
        self.pop_size = int(pop_size)
        self.max_evals = int(max_evals)
        self.update_rounds = (self.max_evals - self.pop_size) // self.pop_size
        self.s_max = float(s_max)
        self.rng = np.random.default_rng(seed)

        self.n_evals = 0
        self.trace: list[CandidateEvaluation] = []
        self.best_history: list[dict[str, Any]] = []
        self.unique_param_keys: set[str] = set()


        self.angle_labels = np.arange(1, 361, dtype=float)
        self.angle_prob = self.angle_labels / self.angle_labels.sum()

    def _decode(self, x: np.ndarray) -> dict[str, Any]:
        params: dict[str, Any] = {}

        for i, name in enumerate(self.names):
            low, high, kind = self.param_spec[name]
            v = float(np.clip(x[i], low, high))

            if kind == "log_cont":
                decoded = float(10.0 ** v)
            elif kind == "int":
                decoded = int(np.clip(np.rint(v), math.ceil(low), math.floor(high)))
            elif kind == "cont":
                decoded = float(v)
            else:
                raise ValueError(f"Unknown parameter kind '{kind}' for {name}.")

            params[name] = decoded

        return params

    def _evaluate(self, x: np.ndarray) -> float:
        if self.n_evals >= self.max_evals:
            raise RuntimeError("SCSO evaluation budget exceeded.")

        params = self._decode(x)
        fitness = float(self.fitness_fn(params))

        self.n_evals += 1
        self.unique_param_keys.add(params_to_json(params))
        self.trace.append(
            CandidateEvaluation(
                eval_id=self.n_evals,
                params=params,
                fitness=fitness,
            )
        )
        return fitness

    def _evaluate_population(self, pop: np.ndarray) -> np.ndarray:
        return np.array([self._evaluate(pop[i]) for i in range(self.pop_size)])

    def _roulette_angle(self) -> float:
        """
        Roulette-wheel selection among 360 direction labels.

        The selected label is used directly in cos(alpha), matching the
        original-style SCSO code family that applies cos() to the roulette
        output rather than introducing a separate model-specific parameter.
        """
        return float(self.rng.choice(self.angle_labels, p=self.angle_prob))

    def _record_best(
        self,
        round_id: int,
        fitness: np.ndarray,
        pop: np.ndarray,
        best_fit: float,
        best_pos: np.ndarray,
    ) -> tuple[float, np.ndarray]:
        idx = int(np.argmax(fitness))
        if fitness[idx] > best_fit:
            best_fit = float(fitness[idx])
            best_pos = pop[idx].copy()

        self.best_history.append(
            {
                "round": int(round_id),
                "n_evals": int(self.n_evals),
                "best_cv_accuracy": float(best_fit),
            }
        )
        return best_fit, best_pos

    def run(self) -> dict[str, Any]:

        pop = self.rng.uniform(
            low=self.lower,
            high=self.upper,
            size=(self.pop_size, self.dim),
        )


        fitness = self._evaluate_population(pop)
        best_idx = int(np.argmax(fitness))
        best_fit = float(fitness[best_idx])
        best_pos = pop[best_idx].copy()
        self.best_history.append(
            {
                "round": 0,
                "n_evals": int(self.n_evals),
                "best_cv_accuracy": float(best_fit),
            }
        )


        for t in range(self.update_rounds):


            T = max(1, self.update_rounds)
            r_G = self.s_max - (self.s_max * t / T)

            old_pop = pop.copy()
            new_pop = np.empty_like(old_pop)

            for i in range(self.pop_size):
                r = r_G * self.rng.random()
                R = 2.0 * r_G * self.rng.random() - r_G


                current = old_pop[i]
                updated = current.copy()

                for j in range(self.dim):
                    alpha = self._roulette_angle()

                    if abs(R) <= 1.0:

                        x_rand = abs(
                            self.rng.random() * best_pos[j] - current[j]
                        )
                        updated[j] = (
                            best_pos[j]
                            - r * x_rand * math.cos(alpha)
                        )
                    else:


                        cp = int(self.rng.integers(0, self.pop_size))
                        candidate = old_pop[cp, j]
                        updated[j] = (
                            r * (
                                candidate
                                - self.rng.random() * current[j]
                            )
                        )

                new_pop[i] = np.clip(updated, self.lower, self.upper)


            pop = new_pop
            fitness = self._evaluate_population(pop)

            best_fit, best_pos = self._record_best(
                round_id=t + 1,
                fitness=fitness,
                pop=pop,
                best_fit=best_fit,
                best_pos=best_pos,
            )

        if self.n_evals != self.max_evals:
            raise RuntimeError(
                f"SCSO ended with {self.n_evals} evaluations; "
                f"expected exactly {self.max_evals}."
            )

        return {
            "best_params": self._decode(best_pos),
            "best_cv_accuracy": float(best_fit),
            "n_evals": int(self.n_evals),
            "n_unique_configs": int(len(self.unique_param_keys)),
            "history": self.best_history,
            "trace": [
                {
                    "eval_id": e.eval_id,
                    "params": e.params,
                    "cv_accuracy": e.fitness,
                }
                for e in self.trace
            ],
        }


def make_cv_accuracy_objective(
    base_pipe: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
):
    """Mean Stratified 5-Fold CV Accuracy — common HPO objective."""
    def objective(params: dict[str, Any]) -> float:
        scores = []

        for train_idx, val_idx in cv_splits:
            pipe = clone(base_pipe)
            pipe.set_params(**params)

            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_val)
            scores.append(accuracy_score(y_val, pred))

        return float(np.mean(scores))

    return objective


def evaluate_best_params_on_cv(
    base_pipe: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
    best_params: dict[str, Any],
    scenario: str,
    model_name: str,
    hpo_name: str,
) -> tuple[dict[str, dict[str, float]], pd.DataFrame]:
    """Return CV mean±SD plus the raw metric row for every fold."""
    fold_rows = []

    for fold_id, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        pipe = clone(base_pipe)
        pipe.set_params(**best_params)

        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        pipe.fit(X_tr, y_tr)

        pred = pipe.predict(X_val)
        score, score_type = continuous_score(pipe, X_val)
        m = compute_metrics(y_val, pred, score)

        fold_rows.append(
            {
                "scenario": scenario,
                "model": model_name,
                "hpo": hpo_name,
                "fold": fold_id,
                "n_train_fold": len(train_idx),
                "n_validation_fold": len(val_idx),
                "score_type": score_type,
                **m,
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = summarize_fold_metrics(fold_df)
    return summary, fold_df


def evaluate_on_heldout_test(
    base_pipe: Pipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    sample_ids_test: pd.Series,
    best_params: dict[str, Any],
    scenario: str,
    model_name: str,
    hpo_name: str,
) -> tuple[dict[str, float], pd.DataFrame]:
    """Refit on full training set once, then evaluate held-out test data."""
    pipe = clone(base_pipe)
    pipe.set_params(**best_params)
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    score, score_type = continuous_score(pipe, X_test)
    metrics = compute_metrics(y_test, pred, score)

    predictions = pd.DataFrame(
        {
            "sample_id": sample_ids_test.astype(str).to_numpy(),
            "scenario": scenario,
            "model": model_name,
            "hpo": hpo_name,
            "true_label": y_test.to_numpy(dtype=int),
            "predicted_label": np.asarray(pred, dtype=int),
            "score": np.asarray(score, dtype=float),
            "score_type": score_type,
        }
    )

    return metrics, predictions


def run_gridsearch(
    base_pipe: Pipeline,
    grid: dict[str, list[Any] | np.ndarray],
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
) -> dict[str, Any]:
    gs = GridSearchCV(
        estimator=base_pipe,
        param_grid=grid,
        scoring="accuracy",
        cv=cv_splits,
        n_jobs=-1,
        refit=False,
        return_train_score=False,
    )
    gs.fit(X, y)

    trace_rows = []
    params_list = gs.cv_results_["params"]
    mean_scores = gs.cv_results_["mean_test_score"]
    rank_scores = gs.cv_results_["rank_test_score"]

    for i, (params, mean_score, rank) in enumerate(
        zip(params_list, mean_scores, rank_scores),
        start=1,
    ):
        trace_rows.append(
            {
                "eval_id": i,
                "params": to_builtin(params),
                "cv_accuracy": float(mean_score),
                "rank": int(rank),
            }
        )

    return {
        "best_params": to_builtin(gs.best_params_),
        "best_cv_accuracy": float(gs.best_score_),
        "n_evals": int(len(params_list)),
        "n_unique_configs": int(
            len({params_to_json(p) for p in params_list})
        ),
        "trace": trace_rows,
    }


def run_scso(
    base_pipe: Pipeline,
    scso_spec: dict[str, tuple[float, float, str]],
    X: pd.DataFrame,
    y: pd.Series,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
) -> dict[str, Any]:
    objective = make_cv_accuracy_objective(base_pipe, X, y, cv_splits)

    optimizer = SCSO(
        param_spec=scso_spec,
        fitness_fn=objective,
        pop_size=SCSO_POP_SIZE,
        max_evals=HPO_BUDGET,
        s_max=SCSO_S_MAX,
        seed=RANDOM_SEED,
    )
    return optimizer.run()




In [ ]:


import hashlib
from datetime import datetime, timezone
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, matthews_corrcoef

SCSO_SEEDS = [42, 52, 62, 72, 82]  # independent optimizer seeds
SELECTED_MODELS = ['LR', 'SVM', 'RF', 'XGB']
SELECTED_SCENARIOS = ['A_baseline', 'B_plus_bia', 'C_plus_lab', 'D_full']
SPLIT_SEED = 42  # tetap: kontrol agar hanya stochastic optimizer yang berubah
CV_SEED = 42
RESULT_FILE = str(RESULTS_DIR / 'scso_multiseed.xlsx')

RUN_COLUMNS = [
    'scso_seed','split_seed','cv_seed','model','scenario','n_features',
    'train_n','test_n','cv_best_accuracy','n_evals','n_unique_configs',
    'optimization_seconds','cv_summary_seconds','refit_and_test_seconds','best_params_json',
    'cv_accuracy_mean','cv_accuracy_sd','cv_f1_mean','cv_f1_sd','cv_roc_auc_mean','cv_roc_auc_sd',
    'test_accuracy','test_precision','test_recall','test_specificity',
    'test_balanced_accuracy','test_f1','test_roc_auc','test_mcc',
    'test_tn','test_fp','test_fn','test_tp'
]


def _clean_table(frame, columns=None):
    if frame is None or len(frame)==0:
        return pd.DataFrame(columns=columns or [])
    return frame.reset_index(drop=True).copy()


def _build_summary(run_df):
    if run_df.empty:
        return pd.DataFrame()
    keys=['model','scenario']
    fields=['cv_best_accuracy','test_accuracy','test_f1','test_roc_auc',
            'test_balanced_accuracy','test_mcc','optimization_seconds','n_unique_configs']
    agg={name:['mean','std','min','max'] for name in fields}
    summary=run_df.groupby(keys,dropna=False).agg(agg)
    summary.columns=['_'.join(k) for k in summary.columns.to_flat_index()]
    summary=summary.reset_index()
    summary.insert(2,'n_seeds',run_df.groupby(keys).size().to_numpy())
    return summary


def _build_contrasts(run_df):
    metrics=['test_accuracy','test_f1','test_roc_auc']
    contrasts={'B-A':('B_plus_bia','A_baseline'),
               'C-A':('C_plus_lab','A_baseline'),
               'D-B':('D_full','B_plus_bia'),
               'D-C':('D_full','C_plus_lab')}
    rows=[]
    for (seed,model), group in run_df.groupby(['scso_seed','model']):
        values=group.set_index('scenario')
        for contrast,(left,right) in contrasts.items():
            if left not in values.index or right not in values.index:
                continue
            r={'scso_seed':int(seed),'model':model,'contrast':contrast}
            for metric in metrics:
                r['delta_'+metric.replace('test_','')]=float(values.loc[left,metric]-values.loc[right,metric])
            rows.append(r)
    return pd.DataFrame(rows)


def _build_contrast_summary(contrast_df):
    if contrast_df.empty:
        return pd.DataFrame()
    vals=['delta_accuracy','delta_f1','delta_roc_auc']
    summary=contrast_df.groupby(['model','contrast'])[vals].agg(['mean','std','min','max'])
    summary.columns=['_'.join(c) for c in summary.columns.to_flat_index()]
    summary=summary.reset_index()
    summary.insert(2,'n_seeds',contrast_df.groupby(['model','contrast']).size().to_numpy())
    return summary


def _save_excel(runs, histories, traces, folds, predictions, metadata):

    contrasts=_build_contrasts(runs)
    sheets={
        'RunMetrics':runs,
        'MeanSD':_build_summary(runs),
        'FeatureContrasts':contrasts,
        'ContrastMeanSD':_build_contrast_summary(contrasts),
        'SCSOConvergence':histories,
        'SCSOFitnessTrace':traces,
        'CVFoldMetrics':folds,
        'TestPredictions':predictions,
        'Metadata':pd.DataFrame([metadata]),
    }
    tmp_file=RESULT_FILE+'.partial.xlsx'
    with pd.ExcelWriter(tmp_file, engine='openpyxl') as writer:
        for sheet_name,table in sheets.items():
            _clean_table(table).to_excel(writer,sheet_name=sheet_name,index=False)
            ws=writer.sheets[sheet_name]
            ws.freeze_panes='A2'
            ws.auto_filter.ref=ws.dimensions
            from openpyxl.styles import Font, PatternFill, Alignment
            for cell in ws[1]:
                cell.font=Font(bold=True,color='FFFFFF')
                cell.fill=PatternFill(fill_type='solid',fgColor='17365D')
                cell.alignment=Alignment(wrap_text=True)
            for col in ws.columns:
                letter=col[0].column_letter
                maximum=max((len(str(c.value or '')) for c in list(col)[:90]),default=12)
                ws.column_dimensions[letter].width=min(42,max(14,maximum+2))
    import os
    os.replace(tmp_file,RESULT_FILE)


def _restore_if_existing(expected_fingerprint):
    if not Path(RESULT_FILE).exists():
        empty=pd.DataFrame()
        return (pd.DataFrame(columns=RUN_COLUMNS),empty,empty,empty,empty)
    meta=pd.read_excel(RESULT_FILE,sheet_name='Metadata')
    if meta.empty or str(meta.iloc[0]['dataset_sha256'])!=expected_fingerprint:
        raise ValueError('Existing results were generated from a different dataset.')
    if int(meta.iloc[0]['split_seed'])!=SPLIT_SEED or int(meta.iloc[0]['cv_seed'])!=CV_SEED:
        raise ValueError('Existing results use different split or CV settings.')
    tables=[pd.read_excel(RESULT_FILE,sheet_name=n) for n in
        ['RunMetrics','SCSOConvergence','SCSOFitnessTrace','CVFoldMetrics','TestPredictions']]
    print(f'Resuming from {len(tables[0])} completed runs.', flush=True)
    return tuple(tables)


def run_scso_multiseed():
    df=load_dataset(DATA_PATH)
    validate_columns_and_data(df)
    y,target_map=encode_binary_target(df[TARGET_COL])
    if len(df)!=319 or df.shape[1]!=39:
        raise ValueError(f'Unexpected dataset shape: {df.shape}')
    fingerprint=hashlib.sha256(pd.util.hash_pandas_object(df,index=True).values.tobytes()).hexdigest()
    all_idx=np.arange(len(df))
    train_idx,test_idx=train_test_split(all_idx,test_size=TEST_SIZE,stratify=y,random_state=SPLIT_SEED)
    y_train=y.iloc[train_idx].reset_index(drop=True)
    y_test=y.iloc[test_idx].reset_index(drop=True)
    cv=list(StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=CV_SEED).split(
        np.zeros(len(y_train)),y_train))
    metadata={
        'dataset_sha256':fingerprint,'n_samples':len(df),'n_train':len(train_idx),'n_test':len(test_idx),
        'split_seed':SPLIT_SEED,'cv_seed':CV_SEED,'seeds_json':json.dumps(SCSO_SEEDS),
        'model_list_json':json.dumps(SELECTED_MODELS),'scenario_list_json':json.dumps(SELECTED_SCENARIOS),
        'hpo_budget':HPO_BUDGET,'population_size':SCSO_POP_SIZE,'s_max':SCSO_S_MAX,
        'raw_positive_label':target_map['raw_positive_label'],
        'machine':__import__('platform').machine(),
        'platform':__import__('platform').platform(),
        'logical_cpu_count':__import__('os').cpu_count(),
        'scikit_learn':__import__('sklearn').__version__,
        'numpy':np.__version__,'pandas':pd.__version__,'xgboost':__import__('xgboost').__version__,
        'experiment_type':'five SCSO optimizer seeds on one FIXED 80/20 split and fixed 5-fold CV',
        'last_updated_utc':datetime.now(timezone.utc).isoformat()
    }
    runs,histories,traces,folds,predictions=_restore_if_existing(fingerprint)
    completed={(int(r['scso_seed']),r['model'],r['scenario']) for _,r in runs.iterrows()}
    total=len(SCSO_SEEDS)*len(SELECTED_MODELS)*len(SELECTED_SCENARIOS)
    print(f'Planned: {total} SCSO runs × 100 evaluations × 5 folds.',flush=True)
    print(f'Fixed split {SPLIT_SEED} CV {CV_SEED}; train {len(train_idx)} test {len(test_idx)}.',flush=True)
    print(f'Checkpoint file: {RESULT_FILE}', flush=True)
    for seed in SCSO_SEEDS:
        for model_name in SELECTED_MODELS:
            spec=MODEL_SPECS[model_name]
            for scenario in SELECTED_SCENARIOS:
                key=(seed,model_name,scenario)
                if key in completed:
                    continue
                cols=SCENARIOS[scenario]
                X_train=df.iloc[train_idx][cols].reset_index(drop=True)
                X_test=df.iloc[test_idx][cols].reset_index(drop=True)
                estimator=clone(spec['estimator'])
                if model_name in ('RF','XGB'):
                    estimator.set_params(n_jobs=1)  # limit parallel workers
                pipeline=make_pipeline(estimator)
                objective=make_cv_accuracy_objective(pipeline,X_train,y_train,cv)
                optimizer=SCSO(param_spec=spec['scso_spec'],fitness_fn=objective,
                    pop_size=SCSO_POP_SIZE,max_evals=HPO_BUDGET,s_max=SCSO_S_MAX,seed=int(seed))
                print(f'[{len(completed)+1}/{total}] Seed {seed} | {model_name} | {scenario} ...',flush=True)
                t0=time.perf_counter()
                result=optimizer.run()
                search_time=time.perf_counter()-t0
                if int(result['n_evals'])!=100 or len(result['history'])!=10:
                    raise RuntimeError('Jumlah evaluasi SCSO atau round history salah.')
                t0=time.perf_counter()
                cv_sum,fold_df=evaluate_best_params_on_cv(pipeline,X_train,y_train,cv,
                    result['best_params'],scenario,model_name,'SCSO')
                cv_time=time.perf_counter()-t0
                t0=time.perf_counter()
                test_metrics,pred_df=evaluate_on_heldout_test(pipeline,X_train,y_train,X_test,
                    y_test,get_sample_ids(df).iloc[test_idx].reset_index(drop=True),
                    result['best_params'],scenario,model_name,'SCSO')
                test_time=time.perf_counter()-t0
                tn,fp,fn,tp=confusion_matrix(pred_df['true_label'],pred_df['predicted_label'],labels=[0,1]).ravel()
                record={
                    'scso_seed':seed,'split_seed':SPLIT_SEED,'cv_seed':CV_SEED,
                    'model':model_name,'scenario':scenario,'n_features':len(cols),
                    'train_n':len(train_idx),'test_n':len(test_idx),
                    'cv_best_accuracy':result['best_cv_accuracy'],
                    'n_evals':result['n_evals'],'n_unique_configs':result['n_unique_configs'],
                    'optimization_seconds':round(search_time,3),
                    'cv_summary_seconds':round(cv_time,3),
                    'refit_and_test_seconds':round(test_time,3),
                    'best_params_json':params_to_json(result['best_params']),
                    'cv_accuracy_mean':cv_sum['accuracy']['mean'],
                    'cv_accuracy_sd':cv_sum['accuracy']['sd'],
                    'cv_f1_mean':cv_sum['f1']['mean'],'cv_f1_sd':cv_sum['f1']['sd'],
                    'cv_roc_auc_mean':cv_sum['roc_auc']['mean'],'cv_roc_auc_sd':cv_sum['roc_auc']['sd'],
                    'test_accuracy':test_metrics['accuracy'],
                    'test_precision':test_metrics['precision'],
                    'test_recall':test_metrics['recall'],
                    'test_specificity':float(tn/(tn+fp)) if tn+fp else np.nan,
                    'test_balanced_accuracy':float(balanced_accuracy_score(pred_df['true_label'],pred_df['predicted_label'])),
                    'test_f1':test_metrics['f1'],'test_roc_auc':test_metrics['roc_auc'],
                    'test_mcc':float(matthews_corrcoef(pred_df['true_label'],pred_df['predicted_label'])),
                    'test_tn':int(tn),'test_fp':int(fp),'test_fn':int(fn),'test_tp':int(tp),
                }
                runs=pd.concat([runs,pd.DataFrame([record])],ignore_index=True)
                rows_history=[{'scso_seed':seed,'model':model_name,'scenario':scenario,**row} for row in result['history']]
                histories=pd.concat([histories,pd.DataFrame(rows_history)],ignore_index=True)
                rows_trace=[{'scso_seed':seed,'model':model_name,'scenario':scenario,
                    'eval_id':t['eval_id'],'cv_accuracy':t['cv_accuracy'],
                    'params_json':params_to_json(t['params'])} for t in result['trace']]
                traces=pd.concat([traces,pd.DataFrame(rows_trace)],ignore_index=True)
                fold_df.insert(0,'scso_seed',seed)
                folds=pd.concat([folds,fold_df],ignore_index=True)
                pred_df.insert(0,'scso_seed',seed)
                predictions=pd.concat([predictions,pred_df],ignore_index=True)
                completed.add(key)
                metadata['last_updated_utc']=datetime.now(timezone.utc).isoformat()
                _save_excel(runs,histories,traces,folds,predictions,metadata)
                print(f'  complete | CV={result["best_cv_accuracy"]:.3f} '
                      f'Acc={test_metrics["accuracy"]:.3f} F1={test_metrics["f1"]:.3f} '
                      f'AUC={test_metrics["roc_auc"]:.3f} | optimization {search_time:.1f} s '
                      f'| saved {len(completed)}/{total}',flush=True)
    print(f'Saved: {RESULT_FILE}', flush=True)

run_scso_multiseed()
